In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql import types as t
from pyspark.sql.window import Window
from datetime import datetime
import logging        
from config import ROUTES, PipelineConfig  

In [0]:
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger(__name__)

In [0]:
# Config 
BRONZE_PATH = "workspace.case_spark_cvm.bronze_registro_fundo_cvm"
NOME_TABELA  = f"silver_registro_fundo_cvm" 
SILVER_PATH = f"{ROUTES.TABLE_BASE}.{NOME_TABELA}"
DATA_PROC    = int(datetime.now().strftime("%Y%m%d"))

## CVM - Fundos Investimentos - Registros Fundos

In [0]:
df_registro_fundo_cvm = PipelineConfig.ler_ultima_particao(spark=spark, table_name=BRONZE_PATH, partition_col="data_processamento")

### 1.1 tratemento silver

#### 1.1.1 Normalizando CNPJ

In [0]:
df_registro_fundo_cvm = df_registro_fundo_cvm.withColumn(
    "CNPJ_Fundo",
    PipelineConfig.normalizar_cnpj("CNPJ_Fundo")
)

#### 1.1.2 Retirando dados duplicados

A origem de dados da CVM é transacional e frequentemente envia múltiplos registros para o mesmo ***CNPJ_Fundo*** ao longo do tempo. Isso não é necessariamente um erro, mas sim o reflexo da evolução do ciclo de vida do fundo (atualizações de cadastro, mudanças de status ou encerramentos).

Para garantir a unicidade estrita desta Tabela Dimensão e refletir a realidade mais atualizada, aplicamos uma lógica de desempate em duas etapas:
1. **Prioridade Temporal:** Mantemos exclusivamente o registro com a ***Data_Registro*** mais recente.
2. **Desempate por Status:** Em caso de colisão de datas (dois registros no mesmo dia), a prioridade recai sobre a situação operacional (pesos maiores para *"Em Funcionamento Normal"* e *"Fase Pré-Operacional"*).

*Nota Arquitetural:* Duplicidades residuais que indicam anomalias severas na origem (ex: múltiplos registros ativos idênticos para o mesmo CNPJ no mesmo momento) são interceptadas e isoladas na tabela de Quarentena para posterior notificação aos responsáveis.

In [0]:
# A função remover_duplicatas usa a ordem desc(). 
# Para que 'Normal' ganhe de 'Cancelado' no mesmo dia, damos a letra 'B' (maior) para status ativos e 'A' para inativos.
df_registro_fundo_cvm = df_registro_fundo_cvm.withColumn(
    "_peso_status",
    f.when(f.col("Situacao").isin("Em Funcionamento Normal", "Fase Pré-Operacional"), f.lit("B"))
     .otherwise(f.lit("A"))
)

# Criamos a super-chave temporal. Ex: "2025-06-27_B" vence de "2025-06-27_A" e de "2024-01-01_B"
df_registro_fundo_cvm = df_registro_fundo_cvm.withColumn(
    "_ordem_desempate",
    f.concat_ws("_", f.col("Data_Registro"), f.col("_peso_status"))
)

#Espalhamos a DATA VENCEDORA para todas as linhas daquele CNPJ
window_fundo = Window.partitionBy("CNPJ_Fundo")

df_registro_fundo_cvm = df_registro_fundo_cvm.withColumn(
    "_max_ordem_desempate", 
    f.max("_ordem_desempate").over(window_fundo)
).withColumn(
    "_data_oficial",
    f.max(
        f.when(f.col("_ordem_desempate") == f.col("_max_ordem_desempate"), f.col("Data_Registro"))
    ).over(window_fundo)
)

# Chamamos a função central do pipeline usando a nossa super-chave
df_registro_fundo_cvm, df_todas_duplicadas = PipelineConfig.remover_duplicatas(
    df=df_registro_fundo_cvm,
    chave_negocio=["CNPJ_Fundo"],
    coluna_ordenacao="_ordem_desempate"
)

# FILTRO ANTI-SPAM (A QUARENTENA REAL):
# Se a linha descartada tem a MESMA DATA da linha oficial, é colisão na origem (erro grave da CVM).
# Se a linha descartada tem data mais antiga, é apenas histórico defasado (descarte silencioso).
df_quarentena_real = (df_todas_duplicadas
    .filter(f.col("Data_Registro") == f.col("_data_oficial"))
    .withColumn("_motivo_quarentena", f.lit("Anomalia CVM: Múltiplos registros conflitantes no mesmo CNPJ para a mesma Data de Registro"))
)

# Limpeza pesada das colunas auxiliares (agora usando a data oficial)
colunas_sujeira = ["_peso_status", "_ordem_desempate", "_data_oficial"]
df_registro_fundo_cvm = df_registro_fundo_cvm.drop(*colunas_sujeira)
df_quarentena_real = df_quarentena_real.drop(*colunas_sujeira)

# Gravação do que realmente importa
PipelineConfig.salvar_quarentena(
    spark=spark,
    df_quarentena=df_quarentena_real, 
    tabela_origem="bronze_registro_fundo_cvm", 
    data_proc=DATA_PROC
)

#### 1.1.3 Retirando dados nulos de Colunas Cores

In [0]:
regras_qualidade = {
    "ID_Registro_Fundo": "not_null",   # Não pode ser vazio (Substitui o dropna)
    "CNPJ_Fundo": "not_null",          # Não pode ser vazio (Substitui o dropna)
    "Codigo_CVM": "not_null",          # Não pode ser vazio (Substitui o dropna)
    "Data_Registro": "not_null",       # Não pode ser vazio (Substitui o dropna)
    "Situacao": "not_null",            # Não pode ser vazio (Substitui o dropna)
}

df_registro_fundo_cvm, df_quarentena = PipelineConfig.aplicar_qualidade_e_separar(
    df=df_registro_fundo_cvm,
    regras=regras_qualidade
    )

PipelineConfig.salvar_quarentena(
    spark=spark,
    df_quarentena=df_quarentena, 
    tabela_origem="bronze_registro_fundo_cvm", 
    data_proc=DATA_PROC
)

#### 1.1.4 Tratamento do Tipo de Dado

In [0]:
# Dropando as colunas de metadados
df_registro_fundo_cvm = df_registro_fundo_cvm.drop("_source_url", "_ingest_timestamp", "data_processamento")


In [0]:
df_registro_fundo_cvm = df_registro_fundo_cvm.select(
    # 1. Chaves de Identificação e Nomes
    f.col('ID_Registro_Fundo').cast(t.IntegerType()).alias('id_registro_fundo'),
    f.col('CNPJ_Fundo').cast(t.StringType()).alias('cnpj_fundo'),
    f.col('Codigo_CVM').cast(t.IntegerType()).alias('codigo_cvm'),
    f.col('Tipo_Fundo').cast(t.StringType()).alias('tipo_fundo'),
    f.col('Denominacao_Social').cast(t.StringType()).alias('denominacao_social'),
    
    # 2. Ciclo de Vida, Datas e Status
    f.col('Data_Registro').cast(t.DateType()).alias('data_registro'),
    f.col('Data_Constituicao').cast(t.DateType()).alias('data_constituicao'),
    f.col('Data_Cancelamento').cast(t.DateType()).alias('data_cancelamento'),
    f.col('Situacao').cast(t.StringType()).alias('situacao'),
    f.col('Data_Inicio_Situacao').cast(t.DateType()).alias('data_inicio_situacao'),
    f.col('Data_Adaptacao_RCVM175').cast(t.DateType()).alias('data_adaptacao_rcvm175'),
    f.col('Data_Inicio_Exercicio_Social').cast(t.DateType()).alias('data_inicio_exercicio_social'),
    f.col('Data_Fim_Exercicio_Social').cast(t.DateType()).alias('data_fim_exercicio_social'),
    
    # 3. Informações Patrimoniais
    f.col('Patrimonio_Liquido').cast(t.DecimalType(25, 2)).alias('patrimonio_liquido'),
    f.col('Data_Patrimonio_Liquido').cast(t.DateType()).alias('data_patrimonio_liquido'),
    
    # 4. Governança e Prestadores de Serviço
    f.col('Diretor').cast(t.StringType()).alias('diretor'),
    f.col('CNPJ_Administrador').cast(t.StringType()).alias('cnpj_administrador'),
    f.col('Administrador').cast(t.StringType()).alias('administrador'),
    f.col('Tipo_Pessoa_Gestor').cast(t.StringType()).alias('tipo_pessoa_gestor'),
    f.col('CPF_CNPJ_Gestor').cast(t.StringType()).alias('cpf_cnpj_gestor'),
    f.col('Gestor').cast(t.StringType()).alias('gestor')
)

### 1.2 Salvar na camada Silver

In [0]:
# Definindo as chaves estrangeiras 
chave_negocio = ["cnpj_fundo"]

PipelineConfig.upsert_silver(
    spark=spark, 
    df_novo=df_registro_fundo_cvm, 
    tabela_destino=SILVER_PATH, 
    chave_negocio=chave_negocio
    )